# Brain Fresh — export TRIBE analysis on Colab (GPU)

Use this when local (Mac) inference is too slow. You get a JSON file that matches the Brain Fresh API.

**Workflow**
1. Runtime → Change runtime type → **GPU** (T4/L4/A100).
2. Run setup cells below (installs the backend + TRIBE; first run takes several minutes).
3. Upload your video (same file you will use in the app). **Tip:** macOS screen recordings often have spaces and a special Unicode space before `AM/PM` in the filename; the export cell copies your file to `/content/brain_input.mp4` to avoid path issues.
4. Set `BRAIN_FRESH_VIDEO_ID` to the **8-character id** shown in the web app right after upload (open DevTools → Network → `upload` response, or use the import flow below).
5. Run export (uses **`--progress`**: a tqdm bar for the 4 checkpoint loads + feature extraction + each predict pass); download `analysis_export.json`.
6. Locally: add `ALLOW_ANALYSIS_IMPORT=1` to `backend/.env`, restart the API, then:
   `curl -X POST "http://localhost:8000/api/video/import-result/YOUR_ID" -H "Content-Type: application/json" -d @analysis_export.json`
7. In the app, if processing failed, use **Load analysis JSON** and pick the file — or refresh and open results if you used `curl`.

Optional: `huggingface-cli login` or set `HF_TOKEN` for higher Hub rate limits.

In [ ]:
!nvidia-smi

In [ ]:
# Clone your fork (replace URL) or upload a zip of brain_fresh and %cd into it
# !git clone https://github.com/YOUR/brain_fresh.git
%cd /content/brain_fresh/backend

!pip install -q uv 2>/dev/null || true
!pip install -e . --quiet

In [ ]:
from google.colab import files
import pathlib

up = files.upload()
video_path = pathlib.Path(list(up.keys())[0]).resolve()
print("Video:", video_path)

In [ ]:
# Paste the video_id from Brain Fresh after upload (8 chars)
BRAIN_FRESH_VIDEO_ID = "xxxxxxxx"

import os
import shutil
import subprocess
import sys
import pathlib
from google.colab import files

BACKEND = pathlib.Path("/content/brain_fresh/backend")
SCRIPT = BACKEND / "scripts" / "export_tribe_analysis.py"
if not SCRIPT.is_file():
    raise FileNotFoundError(
        f"Expected {SCRIPT}. Clone the repo and run the install cell (path must be /content/brain_fresh)."
    )

# Avoid macOS "Screen Recording … AM.mov" unicode (U+202F) / space issues on Linux
safe_video = pathlib.Path("/content/brain_input.mp4")
shutil.copy2(video_path, safe_video)
print("Using:", safe_video)

out = pathlib.Path("/content/analysis_export.json")
cmd = [
    sys.executable,
    str(SCRIPT),
    str(safe_video),
    "-o",
    str(out),
    "--video-id",
    BRAIN_FRESH_VIDEO_ID,
    "--progress",
]
env = {**os.environ, "PYTHONUNBUFFERED": "1"}
print("Running:\n ", " ".join(cmd), "\n")
# Inherit stdout/stderr so tqdm + tracebacks show in the cell output
r = subprocess.run(cmd, cwd=str(BACKEND), env=env)
if r.returncode != 0:
    raise RuntimeError(
        f"export_tribe_analysis.py exited with {r.returncode}. "
        "Scroll up in this cell for the Python traceback (CUDA OOM, missing HF token, etc.)."
    )
files.download(str(out))